# Stock Pattern Multi-Task CNN

Multi-task CNN for simultaneous chart-pattern classification and trend prediction, trained on algorithmically-labeled candlestick images from NASDAQ-100 stocks.

**Sections**
1. Setup & Data Loading
2. Multi-Task CNN (main model)
3. Baseline 1 — Single-Task CNNs
4. Baseline 2 — Random Forest + HOG
5. Evaluation & Comparison

## 1. Setup & Data Loading

In [ ]:
!pip install scikit-image -q

import os, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from skimage.feature import hog

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = "/content/drive/MyDrive/APS360_Stock_Pattern/dataset"
LABELS_CSV = os.path.join(DATA_ROOT, "labels.csv")
IMAGE_DIR = os.path.join(DATA_ROOT, "images")

df_labels = pd.read_csv(LABELS_CSV)
print(f"Total samples: {len(df_labels)}")
print(f"\nPattern distribution:\n{df_labels['pattern'].value_counts()}")
print(f"\nTrend distribution:\n{df_labels['trend'].value_counts()}")
print(f"\nSplit:\n{df_labels['split'].value_counts()}")

In [ ]:
PATTERN_CLASSES = sorted(df_labels["pattern"].unique().tolist())
TREND_CLASSES = sorted(df_labels["trend"].unique().tolist())
NUM_PATTERNS = len(PATTERN_CLASSES)
NUM_TRENDS = len(TREND_CLASSES)

pattern2idx = {c: i for i, c in enumerate(PATTERN_CLASSES)}
trend2idx = {c: i for i, c in enumerate(TREND_CLASSES)}

print(f"Pattern classes ({NUM_PATTERNS}): {PATTERN_CLASSES}")
print(f"Trend classes   ({NUM_TRENDS}):   {TREND_CLASSES}")


class StockPatternDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["filename"])
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        pat_label = pattern2idx[row["pattern"]]
        trend_label = trend2idx[row["trend"]]
        return img, pat_label, trend_label


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

df_train = df_labels[df_labels["split"] == "train"]
df_val = df_labels[df_labels["split"] == "val"]

train_ds = StockPatternDataset(df_train, IMAGE_DIR, train_transform)
val_ds = StockPatternDataset(df_val, IMAGE_DIR, val_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

## 2. Multi-Task CNN

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class Backbone(nn.Module):
    """Shared convolutional feature extractor."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32),    # 224 -> 112
            ConvBlock(32, 64),   # 112 -> 56
            ConvBlock(64, 128),  #  56 -> 28
            ConvBlock(128, 256), #  28 -> 14
        )
        self.pool = nn.AdaptiveAvgPool2d((4, 4))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return x.view(x.size(0), -1)  # (B, 4096)


class TaskHead(nn.Module):
    def __init__(self, in_dim, num_classes, dropout=0.5):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(x)


class MultiTaskCNN(nn.Module):
    def __init__(self, num_patterns, num_trends):
        super().__init__()
        self.backbone = Backbone()
        self.pattern_head = TaskHead(4096, num_patterns)
        self.trend_head = TaskHead(4096, num_trends)

    def forward(self, x):
        features = self.backbone(x)
        return self.pattern_head(features), self.trend_head(features)


model = MultiTaskCNN(NUM_PATTERNS, NUM_TRENDS).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Multi-Task CNN parameters: {total_params:,}")

### Training loop

In [ ]:
EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-4
ALPHA = 1.0   # pattern loss weight
BETA = 1.0    # trend loss weight
PATIENCE = 8  # early-stopping patience

criterion_pat = nn.CrossEntropyLoss()
criterion_trend = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def train_one_epoch(model, loader):
    model.train()
    total_loss, pat_correct, trend_correct, n = 0, 0, 0, 0
    for imgs, pat_labels, trend_labels in loader:
        imgs = imgs.to(device)
        pat_labels = pat_labels.to(device)
        trend_labels = trend_labels.to(device)

        pat_out, trend_out = model(imgs)
        loss = ALPHA * criterion_pat(pat_out, pat_labels) + \
               BETA * criterion_trend(trend_out, trend_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        pat_correct += (pat_out.argmax(1) == pat_labels).sum().item()
        trend_correct += (trend_out.argmax(1) == trend_labels).sum().item()
        n += imgs.size(0)
    return total_loss / n, pat_correct / n, trend_correct / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, pat_correct, trend_correct, n = 0, 0, 0, 0
    for imgs, pat_labels, trend_labels in loader:
        imgs = imgs.to(device)
        pat_labels = pat_labels.to(device)
        trend_labels = trend_labels.to(device)

        pat_out, trend_out = model(imgs)
        loss = ALPHA * criterion_pat(pat_out, pat_labels) + \
               BETA * criterion_trend(trend_out, trend_labels)

        total_loss += loss.item() * imgs.size(0)
        pat_correct += (pat_out.argmax(1) == pat_labels).sum().item()
        trend_correct += (trend_out.argmax(1) == trend_labels).sum().item()
        n += imgs.size(0)
    return total_loss / n, pat_correct / n, trend_correct / n

In [ ]:
history = {"train_loss": [], "val_loss": [],
           "train_pat_acc": [], "val_pat_acc": [],
           "train_trend_acc": [], "val_trend_acc": []}

best_val_loss = float("inf")
best_state = None
wait = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_pa, tr_ta = train_one_epoch(model, train_loader)
    va_loss, va_pa, va_ta = evaluate(model, val_loader)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_pat_acc"].append(tr_pa)
    history["val_pat_acc"].append(va_pa)
    history["train_trend_acc"].append(tr_ta)
    history["val_trend_acc"].append(va_ta)

    print(f"Epoch {epoch:>3d}/{EPOCHS}  "
          f"Loss {tr_loss:.4f}/{va_loss:.4f}  "
          f"PatAcc {tr_pa:.3f}/{va_pa:.3f}  "
          f"TrendAcc {tr_ta:.3f}/{va_ta:.3f}  "
          f"LR {scheduler.get_last_lr()[0]:.2e}")

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)
print(f"\nBest val loss: {best_val_loss:.4f}")

## 3. Baseline 1 — Single-Task CNNs

Two independent CNNs with the **same backbone architecture**, but each trained on only one task.

In [ ]:
class SingleTaskCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = Backbone()
        self.head = TaskHead(4096, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))


def train_single_task(model, loader, criterion, task_idx, epochs=50, patience=8):
    """Train a single-task CNN. task_idx: 1=pattern, 2=trend (DataLoader tuple pos)."""
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_loss, best_state, wait = float("inf"), None, 0
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for ep in range(1, epochs + 1):
        # Train
        model.train()
        t_loss, t_cor, n = 0, 0, 0
        for batch in train_loader:
            imgs, labels = batch[0].to(device), batch[task_idx].to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            opt.zero_grad(); loss.backward(); opt.step()
            t_loss += loss.item() * imgs.size(0)
            t_cor += (out.argmax(1) == labels).sum().item()
            n += imgs.size(0)

        # Val
        model.eval()
        v_loss, v_cor, vn = 0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                imgs, labels = batch[0].to(device), batch[task_idx].to(device)
                out = model(imgs)
                loss = criterion(out, labels)
                v_loss += loss.item() * imgs.size(0)
                v_cor += (out.argmax(1) == labels).sum().item()
                vn += imgs.size(0)

        sched.step()
        tl, vl = t_loss / n, v_loss / vn
        ta, va = t_cor / n, v_cor / vn
        hist["train_loss"].append(tl); hist["val_loss"].append(vl)
        hist["train_acc"].append(ta); hist["val_acc"].append(va)

        if ep % 10 == 0 or ep == 1:
            print(f"  Epoch {ep:>3d}  Loss {tl:.4f}/{vl:.4f}  Acc {ta:.3f}/{va:.3f}")

        if vl < best_loss:
            best_loss, best_state, wait = vl, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stop at epoch {ep}")
                break

    model.load_state_dict(best_state)
    return hist


print("=== Training Single-Task CNN for Pattern Classification ===")
st_pattern = SingleTaskCNN(NUM_PATTERNS).to(device)
hist_st_pat = train_single_task(st_pattern, train_loader, nn.CrossEntropyLoss(), task_idx=1)

print("\n=== Training Single-Task CNN for Trend Prediction ===")
st_trend = SingleTaskCNN(NUM_TRENDS).to(device)
hist_st_trend = train_single_task(st_trend, train_loader, nn.CrossEntropyLoss(), task_idx=2)

## 4. Baseline 2 — Random Forest + HOG

Extract HOG (Histogram of Oriented Gradients) features from grayscale images, then train separate Random Forest classifiers for pattern and trend.

In [ ]:
def extract_hog_features(df_split, image_dir, size=(128, 128)):
    """Extract HOG features for all images in a split."""
    features, pat_labels, trend_labels = [], [], []
    for _, row in df_split.iterrows():
        img_path = os.path.join(image_dir, row["filename"])
        img = Image.open(img_path).convert("L").resize(size)
        img_arr = np.array(img, dtype=np.float64)
        feat = hog(img_arr, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), feature_vector=True)
        features.append(feat)
        pat_labels.append(row["pattern"])
        trend_labels.append(row["trend"])
    return np.array(features), np.array(pat_labels), np.array(trend_labels)


print("Extracting HOG features (train)...")
X_train, y_pat_train, y_trend_train = extract_hog_features(df_train, IMAGE_DIR)
print("Extracting HOG features (val)...")
X_val, y_pat_val, y_trend_val = extract_hog_features(df_val, IMAGE_DIR)
print(f"HOG feature dimension: {X_train.shape[1]}")

print("\n--- RF Pattern Classifier ---")
rf_pattern = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_pattern.fit(X_train, y_pat_train)
rf_pat_pred = rf_pattern.predict(X_val)
print(f"Val accuracy: {accuracy_score(y_pat_val, rf_pat_pred):.4f}")

print("\n--- RF Trend Classifier ---")
rf_trend = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_trend.fit(X_train, y_trend_train)
rf_trend_pred = rf_trend.predict(X_val)
print(f"Val accuracy: {accuracy_score(y_trend_val, rf_trend_pred):.4f}")

## 5. Evaluation & Comparison

### 5a. Collect predictions from all models on validation set

In [ ]:
@torch.no_grad()
def collect_preds(model, loader, multitask=True):
    model.eval()
    pat_preds, trend_preds, pat_true, trend_true = [], [], [], []
    for imgs, pl, tl in loader:
        imgs = imgs.to(device)
        if multitask:
            p_out, t_out = model(imgs)
            pat_preds.extend(p_out.argmax(1).cpu().tolist())
            trend_preds.extend(t_out.argmax(1).cpu().tolist())
        else:
            out = model(imgs)
            pat_preds.extend(out.argmax(1).cpu().tolist())
        pat_true.extend(pl.tolist())
        trend_true.extend(tl.tolist())
    return pat_preds, trend_preds, pat_true, trend_true


@torch.no_grad()
def collect_single_preds(model, loader, task_idx):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        imgs, labels = batch[0].to(device), batch[task_idx]
        out = model(imgs)
        preds.extend(out.argmax(1).cpu().tolist())
        trues.extend(labels.tolist())
    return preds, trues


# Multi-task CNN
mt_pat_pred, mt_trend_pred, pat_true, trend_true = collect_preds(model, val_loader, multitask=True)

# Single-task CNNs
st_pat_pred, st_pat_true = collect_single_preds(st_pattern, val_loader, task_idx=1)
st_trend_pred, st_trend_true = collect_single_preds(st_trend, val_loader, task_idx=2)

# RF predictions were already collected above (rf_pat_pred, rf_trend_pred)
# Convert RF string labels to indices for consistency
rf_pat_pred_idx = [pattern2idx[p] for p in rf_pat_pred]
rf_trend_pred_idx = [trend2idx[t] for t in rf_trend_pred]
rf_pat_true_idx = [pattern2idx[p] for p in y_pat_val]
rf_trend_true_idx = [trend2idx[t] for t in y_trend_val]

results = {
    "Multi-Task CNN": {
        "pat_acc": accuracy_score(pat_true, mt_pat_pred),
        "trend_acc": accuracy_score(trend_true, mt_trend_pred),
        "pat_f1": f1_score(pat_true, mt_pat_pred, average="macro"),
        "trend_f1": f1_score(trend_true, mt_trend_pred, average="macro"),
    },
    "Single-Task CNN": {
        "pat_acc": accuracy_score(st_pat_true, st_pat_pred),
        "trend_acc": accuracy_score(st_trend_true, st_trend_pred),
        "pat_f1": f1_score(st_pat_true, st_pat_pred, average="macro"),
        "trend_f1": f1_score(st_trend_true, st_trend_pred, average="macro"),
    },
    "RF + HOG": {
        "pat_acc": accuracy_score(rf_pat_true_idx, rf_pat_pred_idx),
        "trend_acc": accuracy_score(rf_trend_true_idx, rf_trend_pred_idx),
        "pat_f1": f1_score(rf_pat_true_idx, rf_pat_pred_idx, average="macro"),
        "trend_f1": f1_score(rf_trend_true_idx, rf_trend_pred_idx, average="macro"),
    },
}

res_df = pd.DataFrame(results).T
res_df.columns = ["Pattern Acc", "Trend Acc", "Pattern F1", "Trend F1"]
print(res_df.round(4).to_string())

### 5b. Training curves (Multi-Task CNN)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_range, history["train_loss"], label="Train")
axes[0].plot(epochs_range, history["val_loss"], label="Val")
axes[0].set_title("Combined Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs_range, history["train_pat_acc"], label="Train")
axes[1].plot(epochs_range, history["val_pat_acc"], label="Val")
axes[1].set_title("Pattern Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].plot(epochs_range, history["train_trend_acc"], label="Train")
axes[2].plot(epochs_range, history["val_trend_acc"], label="Val")
axes[2].set_title("Trend Accuracy")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Accuracy")
axes[2].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 5c. Model comparison bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
models = list(results.keys())
x = np.arange(len(models))
width = 0.35

# Pattern task
pat_acc = [results[m]["pat_acc"] for m in models]
pat_f1  = [results[m]["pat_f1"] for m in models]
axes[0].bar(x - width/2, pat_acc, width, label="Accuracy")
axes[0].bar(x + width/2, pat_f1, width, label="Macro F1")
axes[0].set_title("Pattern Classification")
axes[0].set_xticks(x); axes[0].set_xticklabels(models, rotation=15)
axes[0].set_ylim(0, 1); axes[0].legend()
for i, (a, f) in enumerate(zip(pat_acc, pat_f1)):
    axes[0].text(i - width/2, a + 0.02, f"{a:.3f}", ha="center", fontsize=9)
    axes[0].text(i + width/2, f + 0.02, f"{f:.3f}", ha="center", fontsize=9)

# Trend task
trend_acc = [results[m]["trend_acc"] for m in models]
trend_f1  = [results[m]["trend_f1"] for m in models]
axes[1].bar(x - width/2, trend_acc, width, label="Accuracy")
axes[1].bar(x + width/2, trend_f1, width, label="Macro F1")
axes[1].set_title("Trend Prediction")
axes[1].set_xticks(x); axes[1].set_xticklabels(models, rotation=15)
axes[1].set_ylim(0, 1); axes[1].legend()
for i, (a, f) in enumerate(zip(trend_acc, trend_f1)):
    axes[1].text(i - width/2, a + 0.02, f"{a:.3f}", ha="center", fontsize=9)
    axes[1].text(i + width/2, f + 0.02, f"{f:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### 5d. Confusion matrices

In [ ]:
def plot_confusion(y_true, y_pred, class_names, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")


fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Row 1: Pattern confusion matrices
plot_confusion(pat_true, mt_pat_pred, PATTERN_CLASSES,
               "Multi-Task CNN — Pattern", axes[0, 0])
plot_confusion(st_pat_true, st_pat_pred, PATTERN_CLASSES,
               "Single-Task CNN — Pattern", axes[0, 1])
plot_confusion(rf_pat_true_idx, rf_pat_pred_idx, PATTERN_CLASSES,
               "RF + HOG — Pattern", axes[0, 2])

# Row 2: Trend confusion matrices
plot_confusion(trend_true, mt_trend_pred, TREND_CLASSES,
               "Multi-Task CNN — Trend", axes[1, 0])
plot_confusion(st_trend_true, st_trend_pred, TREND_CLASSES,
               "Single-Task CNN — Trend", axes[1, 1])
plot_confusion(rf_trend_true_idx, rf_trend_pred_idx, TREND_CLASSES,
               "RF + HOG — Trend", axes[1, 2])

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

### 5e. Per-class classification reports

In [ ]:
print("=" * 60)
print("MULTI-TASK CNN — Pattern Classification")
print("=" * 60)
print(classification_report(pat_true, mt_pat_pred, target_names=PATTERN_CLASSES))

print("=" * 60)
print("MULTI-TASK CNN — Trend Prediction")
print("=" * 60)
print(classification_report(trend_true, mt_trend_pred, target_names=TREND_CLASSES))

print("=" * 60)
print("SINGLE-TASK CNN — Pattern Classification")
print("=" * 60)
print(classification_report(st_pat_true, st_pat_pred, target_names=PATTERN_CLASSES))

print("=" * 60)
print("SINGLE-TASK CNN — Trend Prediction")
print("=" * 60)
print(classification_report(st_trend_true, st_trend_pred, target_names=TREND_CLASSES))

print("=" * 60)
print("RF + HOG — Pattern Classification")
print("=" * 60)
print(classification_report(y_pat_val, rf_pat_pred))

print("=" * 60)
print("RF + HOG — Trend Prediction")
print("=" * 60)
print(classification_report(y_trend_val, rf_trend_pred))

### 5f. Sample predictions

In [ ]:
inv_pat = {v: k for k, v in pattern2idx.items()}
inv_trend = {v: k for k, v in trend2idx.items()}

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
indices = random.sample(range(len(val_ds)), 15)

for ax_idx, i in enumerate(indices):
    row, col = divmod(ax_idx, 5)
    img, pt, tt = val_ds[i]
    img_display = img.permute(1, 2, 0).numpy()
    img_display = (img_display * np.array([0.229, 0.224, 0.225]) +
                   np.array([0.485, 0.456, 0.406]))
    img_display = np.clip(img_display, 0, 1)

    axes[row, col].imshow(img_display)
    axes[row, col].axis("off")

    pred_pat = inv_pat[mt_pat_pred[i]] if i < len(mt_pat_pred) else "?"
    pred_trend = inv_trend[mt_trend_pred[i]] if i < len(mt_trend_pred) else "?"
    true_pat = inv_pat[pt]
    true_trend = inv_trend[tt]

    color = "green" if (pred_pat == true_pat and pred_trend == true_trend) else "red"
    axes[row, col].set_title(
        f"T: {true_pat[:12]}/{true_trend}\n"
        f"P: {pred_pat[:12]}/{pred_trend}",
        fontsize=8, color=color
    )

plt.suptitle("Sample Predictions (Multi-Task CNN)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()